In [95]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [96]:
df = pd.read_csv("../dataset/job_postings_exploration.csv")

In [97]:
df.shape

(17645, 20)

In [98]:
df.head()

,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,in_balanced_dataset,description_length,combined_text
0,Marketing Intern,"US, NY, New York",Marketing,NaN,"<h3>We're Food52, and we've created a groundbr...","<p>Food52, a fast-growing, James Beard Award-w...",<ul>\r\n<li>Experience with content management...,NaN,f,t,f,Other,Internship,NaN,NaN,Marketing,f,f,1002,"Marketing Intern <h3>We're Food52, and we've c..."
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"<h3>90 Seconds, the worlds Cloud Video Product...",<p>Organised - Focused - Vibrant - Awesome!<br...,<p><b>What we expect from you:</b></p>\r\n<p>Y...,<h3><b>What you will get from us</b></h3>\r\n<...,f,t,f,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,f,f,2175,Customer Service - Cloud Video Production <h3>...
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,<h3></h3>\r\n<p>Valor Services provides Workfo...,"<p>Our client, located in Houston, is actively...",<ul>\r\n<li>Implement pre-commissioning and co...,NaN,f,t,f,NaN,NaN,NaN,NaN,NaN,f,f,362,Commissioning Machinery Assistant (CMA) <h3></...
3,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,<p>Our passion for improving quality of life t...,<p><b>THE COMPANY: ESRI – Environmental System...,<ul>\r\n<li>\r\n<b>EDUCATION: </b>Bachelor’s o...,<p>Our culture is anything but corporate—we ha...,f,t,f,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,f,f,2858,Account Executive - Washington DC <p>Our passi...
4,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,<p>SpotSource Solutions LLC is a Global Human ...,<p><b>JOB TITLE:</b> Itemization Review Manage...,<p><b>QUALIFICATIONS:</b></p>\r\n<ul>\r\n<li>R...,<p>Full Benefits Offered</p>,f,t,t,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,f,f,1872,Bill Review Manager <p>SpotSource Solutions LL...


In [99]:
df["fraudulent"].value_counts()

fraudulent
f    16787
t      858
Name: count, dtype: int64

In [100]:
df["fraudulent"].value_counts(normalize=True) * 100

fraudulent
f    95.137433
t     4.862567
Name: proportion, dtype: float64

In [101]:
data = df.copy()

In [102]:
data = data.drop(
    columns=["description_length", "combined_text", "has_salary"],
    errors="ignore"
)

In [103]:
data.columns

Index(['title', 'location', 'department', 'salary_range', 'company_profile',
       'description', 'requirements', 'benefits', 'telecommuting',
       'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent', 'in_balanced_dataset'],
      dtype='str')

In [104]:
data.columns

Index(['title', 'location', 'department', 'salary_range', 'company_profile',
       'description', 'requirements', 'benefits', 'telecommuting',
       'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent', 'in_balanced_dataset'],
      dtype='str')

In [105]:
text_columns = [
    "title",
    "description",
    "requirements",
    "company_profile",
    "benefits"
]

In [106]:
def clean_text(text):
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    
    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)
    
    # Keep alphabetic characters and spaces
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [107]:
for col in text_columns:
    data[col] = data[col].apply(clean_text)

In [108]:
data[text_columns].head()

,title,description,requirements,company_profile,benefits
0,marketing intern,food a fast growing james beard award winning ...,experience with content management systems a m...,we re food and we ve created a groundbreaking ...,nan
1,customer service cloud video production,organised focused vibrant awesome do you have ...,what we expect from you your key responsibilit...,seconds the worlds cloud video production serv...,what you will get from us through being part o...
2,commissioning machinery assistant cma,our client located in houston is actively seek...,implement pre commissioning and commissioning ...,valor services provides workforce solutions th...,nan
3,account executive washington dc,the company esri environmental systems researc...,education bachelor s or master s in gis busine...,our passion for improving quality of life thro...,our culture is anything but corporate we have ...
4,bill review manager,job title itemization review manager location ...,qualifications rn license in the state of texa...,spotsource solutions llc is a global human cap...,full benefits offered


In [109]:
data["combined_text"] = (
    data["title"] + " " +
    data["description"] + " " +
    data["requirements"] + " " +
    data["company_profile"] + " " +
    data["benefits"]
)

In [110]:
data["combined_text"].head()

0    marketing intern food a fast growing james bea...
1    customer service cloud video production organi...
2    commissioning machinery assistant cma our clie...
3    account executive washington dc the company es...
4    bill review manager job title itemization revi...
Name: combined_text, dtype: str

In [111]:
data["description_length"] = data["description"].str.len()

In [112]:
data["description_length"].describe()

count    17645.000000
mean      1209.186285
std       1208.013440
min          0.000000
25%        580.000000
50%        984.000000
75%       1543.000000
max      37767.000000
Name: description_length, dtype: float64

In [113]:
data["has_salary"] = (
    data["salary_range"]
    .fillna("")
    .str.strip()
    .ne("")
    .astype(int)
)

In [114]:
data["has_salary"].value_counts()

has_salary
0    14813
1     2832
Name: count, dtype: int64

In [115]:
X_text = data["combined_text"]

In [116]:
categorical_features = [
    "location",
    "employment_type",
    "industry"
]

In [117]:
numerical_features = [
    "description_length",
    "has_salary"
]

In [118]:
y = data["fraudulent"]

In [119]:
X_train, X_test, y_train, y_test = train_test_split(
    data,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [120]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 14116
Testing samples: 3529


In [121]:
print("\nTraining distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True))


Training distribution:
fraudulent
f    0.951403
t    0.048597
Name: proportion, dtype: float64

Testing distribution:
fraudulent
f    0.951261
t    0.048739
Name: proportion, dtype: float64


In [122]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

In [123]:
X_train_text = tfidf.fit_transform(X_train["combined_text"])

In [124]:
X_test_text = tfidf.transform(X_test["combined_text"])

In [125]:
print("Training TF-IDF shape:", X_train_text.shape)
print("Testing TF-IDF shape:", X_test_text.shape)

Training TF-IDF shape: (14116, 10000)
Testing TF-IDF shape: (3529, 10000)


In [126]:
feature_names = tfidf.get_feature_names_out()

print(feature_names[:100])

['aa' 'aabbf' 'aabbf ca' 'ab' 'ab da' 'ab ea' 'abc' 'abc supply'
 'abilities' 'ability' 'ability and' 'ability to' 'able' 'able to' 'about'
 'about and' 'about creating' 'about it' 'about our' 'about the'
 'about us' 'about what' 'about you' 'about your' 'above' 'above all'
 'above and' 'abroad' 'abroad conversational' 'abroad play' 'abroad see'
 'absolutely' 'ac' 'ac ca' 'ac month' 'academic' 'accelerate' 'accent'
 'accent lsdexception' 'accept' 'acceptable' 'acceptance' 'access'
 'access to' 'accessible' 'accommodation' 'accommodations'
 'accommodations may' 'accommodations willingness' 'accomplish'
 'accordance' 'accordance with' 'according' 'according to' 'accordingly'
 'account' 'account executive' 'account management' 'account manager'
 'account managers' 'accountabilities' 'accountability' 'accountable'
 'accountable for' 'accountant' 'accounting' 'accounting and' 'accounts'
 'accounts and' 'accounts payable' 'accredited' 'accuracy' 'accuracy and'
 'accuracy of' 'accurate' 'accu

In [127]:
encoder = OneHotEncoder(
    handle_unknown="ignore"
)

In [128]:
X_train_cat = encoder.fit_transform(
    X_train[categorical_features]
)

In [129]:
X_test_cat = encoder.transform(
    X_test[categorical_features]
)

In [130]:
scaler = StandardScaler()

In [131]:
X_train_num = scaler.fit_transform(
    X_train[numerical_features]
)

In [132]:
X_test_num = scaler.transform(
    X_test[numerical_features]
)

In [133]:
from scipy.sparse import hstack

In [134]:
X_train_final = hstack([
    X_train_text,
    X_train_cat,
    X_train_num
])

In [135]:
X_test_final = hstack([
    X_test_text,
    X_test_cat,
    X_test_num
])

In [136]:
print("Final training shape:", X_train_final.shape)
print("Final testing shape:", X_test_final.shape)

Final training shape: (14116, 12899)
Final testing shape: (3529, 12899)


In [137]:
import joblib

In [138]:
joblib.dump(
    tfidf,
    "../models/tfidf_vectorizer.pkl"
)

['../models/tfidf_vectorizer.pkl']

In [139]:
joblib.dump(
    encoder,
    "../models/onehot_encoder.pkl"
)

['../models/onehot_encoder.pkl']

In [140]:
joblib.dump(
    scaler,
    "../models/standard_scaler.pkl"
)

['../models/standard_scaler.pkl']

In [141]:
print("X_train:", X_train_final.shape)
print("X_test :", X_test_final.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (14116, 12899)
X_test : (3529, 12899)
y_train: (14116,)
y_test : (3529,)
